# 🎙️ 100% Realistic Neural AI Female Voice Changer (Google Colab GPU)

This notebook runs **W-Okada's Real-Time RVC (Retrieval-based Voice Conversion)** engine on Google's free T4 GPU.
Unlike basic pitch-shifters (which sound fake or robotic), this **deep neural network replaces your vocal cords with actual recorded female voice actors** in real-time.

---
### ⚡ 2-Step Quick Start:
1. In the top menu, click **Runtime** > **Change runtime type** > ensure **T4 GPU** is selected.
2. Click **Runtime** > **Run all** (or run each cell below in order).
3. Click the generated **Cloudflare Tunnel URL** to open the Voice Changer Web Studio directly in your browser!

In [ ]:
# Step 1: Verify NVIDIA GPU
!nvidia-smi

In [ ]:
# Step 2: Clone W-Okada Voice Changer & Install AI Vocoder Dependencies
import os
if not os.path.exists("/content/voice-changer"):
    print("Cloning W-Okada Real-Time AI Voice Changer...")
    !git clone --depth 1 https://github.com/w-okada/voice-changer.git /content/voice-changer

%cd /content/voice-changer/server
!pip install -q pyworld fairseq faiss-gpu pyngrok onnxruntime-gpu soundfile
print("Dependencies installed successfully!")

In [ ]:
# Step 3: Install Cloudflare Tunnel (100% Free, No Sign-up, No Token Needed)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print("Cloudflare Tunnel ready!")

In [ ]:
# Step 4: Launch Real-Time AI Female Voice Server & Expose via Cloudflare
import subprocess
import time
import re
import threading

PORT = 18888

# Start Cloudflare tunnel in background
tunnel_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

print("=" * 65)
print("Waiting for your Public Tunnel URL...")

public_url = None
for line in tunnel_proc.stdout:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        print("=" * 65)
        print("🎉 SUCCESS! YOUR REAL-TIME AI FEMALE VOICE STUDIO IS READY!")
        print("=" * 65)
        print(f"👉 CLICK THIS LINK TO OPEN YOUR AI VOICE STUDIO:\n\n    {public_url}\n")
        print("=" * 65)
        break

# Launch W-Okada Server with GPU acceleration
!python3 MMVCServerSIO.py \
  -p {PORT} \
  --https False \
  --content_vec_500 pretrain/checkpoint_best_legacy_500.pt \
  --hubert_base pretrain/hubert_base.pt \
  --hubert_soft pretrain/hubert/hubert-soft-0d54a1f4.pt \
  --nsf_hifigan pretrain/nsf_hifigan/model \
  --crepe_onnx_full pretrain/crepe_onnx_full.onnx \
  --crepe_onnx_tiny pretrain/crepe_onnx_tiny.onnx \
  --rmvpe pretrain/rmvpe.pt \
  --model_dir model_dir \
  --samples samples.json